# **Одна chain rule: проверяем теорему эквивалентности**

Практика к модулю [«Атрибуция от аксиом»](https://ai-interpretability.school).

Урок утверждает вещь сильную и неожиданную: **Gradient×Input, $\varepsilon$-LRP и DeepLIFT
(Rescale) при трех условиях вычисляют в точности одно и то же число.** Три метода из трех
разных статей, с тремя разными мотивировками.

Это единственное утверждение курса, которое можно проверить **до последнего знака**. Не «карты
похожи», не «корреляция высокая» — а буквально одно и то же число. Проверим.

Заодно ответим на вопрос, который урок оставляет открытым: *почему* у такой сети полнота
получается даром.

Считает мгновенно: сеть маленькая, обучать ее не нужно.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
DIMS = [5, 8, 6, 3]     # размеры слоев: вход, два скрытых, выход
TARGET = 1              # класс, для которого считаем атрибуцию


def build(bias, act=nn.ReLU):
    """Сеть-игрушка. Обучать ее не надо: вся суть в правилах распространения, а не в весах."""
    layers = []
    for a, b in zip(DIMS, DIMS[1:]):
        layers += [nn.Linear(a, b, bias=bias), act()]
    return nn.Sequential(*layers[:-1]).eval()


def trace(net, x):
    """Активации до и после каждого слоя — они нужны всем трем методам."""
    acts = [x]
    for layer in net:
        x = layer(x)
        acts.append(x)
    return acts


x = torch.rand(1, 5) + 0.2
zero = torch.zeros(1, 5)
print(f'вход:', x.numpy().round(3))

## 1. Три метода, три десятка строк

Ключ к тому, чтобы увидеть эквивалентность, — написать все три **самостоятельно**, а не звать
библиотеку. Библиотека спрячет ровно то место, ради которого все затевалось: чем каждый метод
заменяет производную нелинейности.

Смотрите на нелинейность в каждой из трех функций:

- `grad_x_input` — производную считает `backward()`, для ReLU это индикатор $[z>0]$;
- `eps_lrp` — релевантность идет через нелинейность **насквозь**, перераспределяет ее только
  линейный слой;
- `deeplift_rescale` — секущая $\dfrac{g(z)-g(z^0)}{z-z^0}$.

Три разных правила. Сейчас увидим, что на ReLU они совпадают.

In [ ]:
def grad_x_input(net, x, c):
    """Gradient x Input: градиент по входу, домноженный на сам вход."""
    x = x.clone().requires_grad_(True)
    net(x)[0, c].backward()
    return (x * x.grad).detach()


def eps_lrp(net, x, c, eps=1e-9):
    """eps-LRP: релевантность течет назад по слоям, знаменатель со стабилизатором."""
    acts = trace(net, x)
    R = torch.zeros_like(acts[-1])
    R[0, c] = acts[-1][0, c]                 # релевантность выхода равна самому логиту
    for i in range(len(net) - 1, -1, -1):
        layer, a_in = net[i], acts[i]
        if isinstance(layer, nn.Linear):
            z = layer(a_in)
            R = a_in * ((R / (z + eps * torch.sign(z) + (z == 0) * eps)) @ layer.weight)
        # нелинейность релевантность не перераспределяет: она идет насквозь
    return R.detach()


def deeplift_rescale(net, x, baseline, c):
    """DeepLIFT (Rescale): множители-секущие, цепное правило, домножение на отклонение входа."""
    a_x, a_0 = trace(net, x), trace(net, baseline)
    m = torch.zeros_like(a_x[-1])
    m[0, c] = 1.0
    for i in range(len(net) - 1, -1, -1):
        layer = net[i]
        if isinstance(layer, nn.Linear):
            m = m @ layer.weight             # на линейном слое множитель просто идет через веса
        else:
            dz, dx = a_x[i + 1] - a_0[i + 1], a_x[i] - a_0[i]
            safe = torch.where(dx.abs() > 1e-7, dx, torch.ones_like(dx))
            m = m * torch.where(dx.abs() > 1e-7, dz / safe, torch.zeros_like(dx))
    return (m * (x - baseline)).detach()

## 2. Условия выполнены — сверяем числа

In [ ]:
def compare(title, net, x, baseline):
    g = grad_x_input(net, x, TARGET)
    l = eps_lrp(net, x, TARGET)
    d = deeplift_rescale(net, x, baseline, TARGET)
    delta = net(x)[0, TARGET].item() - net(baseline)[0, TARGET].item()
    print(f'{title}')
    print(f'   Grad x Input против eps-LRP    макс. расхождение {(g - l).abs().max():.2e}')
    print(f'   Grad x Input против DeepLIFT   макс. расхождение {(g - d).abs().max():.2e}')
    print(f'   сумма атрибуций {g.sum():.6f}   f(x) - f(baseline) {delta:.6f}')


compare(f'Условия теоремы выполнены: ReLU, без смещений, нулевой baseline', build(bias=False), x, zero)

**Смотрите на вторую строку: расхождение ровно ноль.** Не «мало», не «в пределах
точности» — `0.00e+00`, побитовое совпадение. Grad×Input и DeepLIFT с нулевым baseline на
ReLU-сети без смещений — это буквально одно и то же вычисление, записанное разными словами.

У $\varepsilon$-LRP расхождение порядка $10^{-8}$ — это стабилизатор $\varepsilon$, который мы
поставили не нулевым, чтобы не делить на ноль. Устремите его к нулю — уйдет и оно.

**И третья строка: сумма атрибуций в точности равна $f(x) - f(x')$.** Полнота, ради которой
строился Integrated Gradients, здесь получилась сама собой, у метода, который ее не обещал.
Почему — в разделе 4.

**Задание 1.** Поменяйте `TARGET` на другой класс и `torch.manual_seed` на другое число.
Сохраняется ли побитовое совпадение? А порядок расхождения у $\varepsilon$-LRP?

In [ ]:
# Ваш код здесь

## 3. Ломаем условия по одному

Урок называет три условия: нелинейности кусочно-линейны и проходят через ноль, baseline
нулевой, в сети нет смещений. Проверим каждое — сломаем по одному и посмотрим, что именно
разойдется.

In [ ]:
compare(f'Сломано условие 3: у слоев появились смещения', build(bias=True), x, zero)
compare(f'Сломано условие 1: tanh вместо ReLU', build(bias=False, act=nn.Tanh), x, zero)
compare(f'Сломано условие 2: baseline не нулевой', build(bias=False), x, torch.rand(1, 5) * 0.3)

Читаем результат по строкам, и каждая говорит свое.

**Смещения.** Расходится DeepLIFT, а $\varepsilon$-LRP с Grad×Input по-прежнему совпадает. Но
самое важное в третьей строке: **полнота сломалась** — сумма атрибуций больше не равна
отклонению выхода, и расхождение не маленькое. Причина содержательная: смещение дает вклад
в выход, но **не принадлежит ни одному входному признаку**. Разложить выход по признакам
без остатка стало нечем.

**tanh вместо ReLU.** Расходятся оба метода. Ожидаемо: секущая и касательная у гладкой функции
не совпадают, а отношение $g(z)/z$ у tanh не равно ни той, ни другой.

**Ненулевой baseline.** Расходится DeepLIFT — он единственный из трех, кто baseline вообще
использует. $\varepsilon$-LRP не заметил смены baseline потому, что не знает о его
существовании: у него точка отсчета зашита в правило и всегда нулевая.

**Задание 2.** Сломайте два условия сразу — например, смещения и tanh. Складываются ли
расхождения, или одно перекрывает другое? И отдельный вопрос: при каком из трех нарушений
карта изменится **содержательно**, а не численно?

In [ ]:
# Ваш код здесь

## 4. Почему полнота получилась даром

Урок говорит, что у такой сети сумма Grad×Input равна выходу, но не говорит, откуда это
берется. Проверим догадку.

Сеть из ReLU без смещений обладает особым свойством: умножьте вход на положительное число —
и выход умножится на то же число. Такие функции называют **положительно однородными первой
степени**.

In [ ]:
net = build(bias=False)
base = net(x)[0, TARGET].item()
for k in (0.5, 1.0, 2.0, 3.0):
    print(f'   f({k}x) / f(x) = {net(k * x)[0, TARGET].item() / base:.4f}   ждем {k}')

Совпадает до четвертого знака при любом $k$. Это не случайность: ReLU без смещения
удовлетворяет $\mathrm{ReLU}(kz) = k\,\mathrm{ReLU}(z)$ при $k>0$, а линейный слой без
смещения — тем более. Свойство проходит через всю сеть насквозь.

А для однородной функции первой степени работает **теорема Эйлера об однородных функциях**:

$$\sum_i x_i \frac{\partial f}{\partial x_i} = f(x)$$

Слева — ровно сумма атрибуций Gradient×Input. Справа — выход сети.

**Отсюда важный вывод, который стоит унести.** Полнота у Gradient×Input на такой сети — это
**свойство сети, а не свойство метода**. Стоит добавить смещения, и она исчезает, что мы
и видели в разделе 3. У Integrated Gradients полнота другая: она доказана для **любой**
дифференцируемой модели и любого baseline. Это и есть разница между «повезло» и «гарантировано»,
ради которой писался весь модуль.

**Задание 3.** Убедитесь в этом сами: возьмите сеть **со** смещениями и проверьте однородность
тем же способом. Затем посчитайте для нее сумму Grad×Input и сравните с $f(x)-f(0)$.
Насколько велико расхождение и растет ли оно с числом слоев?

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **Теорема не фигура речи.** Grad×Input и DeepLIFT при выполненных условиях совпадают
  побитово. Если у вас они разошлись — ищите ошибку в коде, а не содержательное различие.
  Это тот самый практический вывод, который дает теорема эквивалентности.
- **Каждое условие отвечает за свое.** Смещения ломают полноту, гладкая нелинейность разводит
  секущую с касательной, ненулевой baseline двигает только DeepLIFT. Знать, какое условие
  нарушено, полезнее, чем знать, что методы разошлись.
- **Полнота бывает даром, и это опасно.** На ReLU-сети без смещений она следует из
  однородности сети, а не из свойств метода. Перенеся тот же метод на сеть со смещениями,
  вы тихо теряете гарантию и не узнаете об этом — если не проверите сумму.
- **Выбирать надо не метод, а два решения:** правило для нелинейности и baseline. Все
  остальное — следствие.